# 3장. 데이터의 첫인상 읽기

이 노트북은 강의안 `book/chapters/ch03_data_first_impression.md`를 따라가며 직접 실행해 보는 실습용 자료입니다.

이번 장의 목표는 멋진 분석 결과를 바로 만드는 것이 아니라, 분석 전에 데이터가 어떤 모양인지 차분히 확인하는 습관을 만드는 것입니다. 코드를 한 셀씩 실행하면서 출력 결과를 보고, 바로 아래 설명과 질문에 답해 보세요.


## 0. 이번 장에서 확인할 것

데이터를 처음 열었을 때는 다음 질문에 답할 수 있어야 합니다.

- 데이터 파일은 몇 개인가?
- 각 파일은 어떤 역할을 하는가?
- 각 파일은 몇 행, 몇 열로 구성되어 있는가?
- 어떤 컬럼이 있고, 각 컬럼은 어떤 의미를 가지는가?
- 숫자, 문자, 날짜 컬럼은 무엇인가?
- 비어 있는 값이나 중복된 값은 없는가?
- 여러 파일을 연결할 수 있는 기준 컬럼은 무엇인가?
- LLM이 설명한 데이터 구조가 실제 데이터와 일치하는가?


![CSV 파일을 pandas DataFrame으로 불러오는 흐름](../book/assets/images/ch03/ch03_csv_to_dataframe_flow.svg)

CSV 파일은 텍스트 파일이지만, pandas로 불러오면 행과 열을 가진 `DataFrame`으로 다룰 수 있습니다.


## 1. 실습 준비

먼저 필요한 패키지를 불러오고, 프로젝트 폴더와 데이터 폴더를 찾습니다. 노트북을 `notebooks` 폴더에서 실행해도 되고, 프로젝트 루트에서 실행해도 되도록 `find_project_root()` 함수를 사용합니다.


In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)


def find_project_root(start: Path) -> Path:
    """현재 위치에서 위로 올라가며 data/raw 폴더가 있는 프로젝트 루트를 찾습니다."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw").exists() and (candidate / "book").exists():
            return candidate
    raise FileNotFoundError("프로젝트 루트를 찾지 못했습니다. 노트북을 저장소 안에서 실행해 주세요.")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data" / "raw"

print("프로젝트 루트:", PROJECT_ROOT)
print("데이터 폴더:", DATA_DIR)


프로젝트 루트: /Users/kjj/MyLLMGit/llm-data-analysis-study
데이터 폴더: /Users/kjj/MyLLMGit/llm-data-analysis-study/data/raw


## 2. 데이터 파일이 있는지 확인하기

파일 경로 오류는 초보자가 가장 자주 만나는 오류입니다. 데이터를 불러오기 전에 필요한 CSV 파일이 실제로 있는지 먼저 확인합니다.


In [2]:
expected_files = [
    "customers.csv",
    "products.csv",
    "orders.csv",
    "order_items.csv",
]

file_check = pd.DataFrame({
    "file": expected_files,
    "path": [str(DATA_DIR / filename) for filename in expected_files],
    "exists": [(DATA_DIR / filename).exists() for filename in expected_files],
})

file_check


,file,path,exists
0,customers.csv,/Users/kjj/MyLLMGit/llm-data-analysis-study/da...,True
1,products.csv,/Users/kjj/MyLLMGit/llm-data-analysis-study/da...,True
2,orders.csv,/Users/kjj/MyLLMGit/llm-data-analysis-study/da...,True
3,order_items.csv,/Users/kjj/MyLLMGit/llm-data-analysis-study/da...,True


`exists`가 모두 `True`이면 다음 단계로 진행할 수 있습니다.

하나라도 `False`라면 데이터가 아직 생성되지 않았을 수 있습니다. 그 경우 터미널에서 아래 명령을 실행해 샘플 데이터를 생성합니다.

```bash
python scripts/generate_sample_data.py
```


## 3. CSV 파일을 DataFrame으로 불러오기

이제 4개의 CSV 파일을 pandas `DataFrame`으로 불러옵니다. 각 변수 이름은 파일 이름과 비슷하게 맞춰 두면 이후 코드를 읽기 쉽습니다.


In [3]:
customers = pd.read_csv(DATA_DIR / "customers.csv")
products = pd.read_csv(DATA_DIR / "products.csv")
orders = pd.read_csv(DATA_DIR / "orders.csv")
order_items = pd.read_csv(DATA_DIR / "order_items.csv")

print("customers:", type(customers))
print("products:", type(products))
print("orders:", type(orders))
print("order_items:", type(order_items))


customers: <class 'pandas.DataFrame'>
products: <class 'pandas.DataFrame'>
orders: <class 'pandas.DataFrame'>
order_items: <class 'pandas.DataFrame'>


분석할 데이터셋이 여러 개일 때는 딕셔너리로 묶어 두면 반복 점검을 하기 편합니다.


In [4]:
datasets = {
    "customers": customers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
}

list(datasets.keys())


['customers', 'products', 'orders', 'order_items']

## 4. 각 파일의 역할 이해하기

이번 과정에서 사용하는 데이터는 가상의 온라인 쇼핑몰 운영 데이터입니다.

| 파일 | 역할 | 먼저 확인할 것 |
| --- | --- | --- |
| `customers.csv` | 고객 정보 | 고객 수, 연령, 성별, 지역, 가입일 |
| `products.csv` | 상품 정보 | 상품 수, 카테고리, 가격 |
| `orders.csv` | 주문 정보 | 주문 수, 주문일, 결제수단, 주문상태 |
| `order_items.csv` | 주문 상세 정보 | 주문별 상품, 수량, 단가 |

처음에는 파일을 합치지 말고, 각 파일을 따로 살펴보는 것이 좋습니다.


![pandas DataFrame 구조 예시](../book/assets/images/ch03/ch03_dataframe_structure.svg)

DataFrame은 행(row)과 열(column)로 구성됩니다. `shape`, `head()`, `columns`, `info()` 같은 기본 도구를 사용해 구조를 확인합니다.


## 5. 데이터 크기 확인하기

`shape`는 데이터의 행과 열 개수를 알려 줍니다.

- 앞 숫자: 행 개수
- 뒤 숫자: 열 개수

예를 들어 `(150, 6)`은 150행 6열이라는 뜻입니다.


In [5]:
print("customers:", customers.shape)
print("products:", products.shape)
print("orders:", orders.shape)
print("order_items:", order_items.shape)


customers: (150, 6)
products: (100, 4)
orders: (300, 5)
order_items: (764, 5)


In [6]:
shape_summary = pd.DataFrame([
    {
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
    }
    for name, df in datasets.items()
])

shape_summary


,dataset,rows,columns
0,customers,150,6
1,products,100,4
2,orders,300,5
3,order_items,764,5


### 생각해 보기

- 가장 행이 많은 데이터셋은 무엇인가요?
- `order_items`가 `orders`보다 행이 많다면, 그 이유는 무엇일까요?
- 분석 보고서에 데이터 규모를 설명한다면 어떤 문장으로 쓸 수 있을까요?


## 6. 데이터 앞부분과 마지막 부분 보기

`head()`는 앞부분 5행을 보여 줍니다. 컬럼명이 예상과 맞는지, 값의 형태가 자연스러운지 빠르게 확인할 때 사용합니다.


In [7]:
customers.head()


,customer_id,name,gender,age,city,signup_date
0,1,김수민,F,19,광주,2024-05-17
1,2,장춘자,F,32,대구,2023-12-20
2,3,김상현,F,61,성남,2025-03-22
3,4,김재호,F,55,울산,2025-05-05
4,5,최준서,F,19,부산,2023-09-23


In [8]:
products.head()


,product_id,product_name,category,price
0,1,전자기기 상품 001,전자기기,160000
1,2,도서 상품 002,도서,34000
2,3,전자기기 상품 003,전자기기,152000
3,4,생활용품 상품 004,생활용품,70000
4,5,식품 상품 005,식품,186000


In [9]:
orders.head()


,order_id,customer_id,order_date,payment_method,order_status
0,1,123,2025-10-16,card,completed
1,2,77,2026-06-05,naver_pay,cancelled
2,3,138,2026-03-12,bank_transfer,cancelled
3,4,57,2026-06-19,kakao_pay,cancelled
4,5,125,2026-05-25,card,cancelled


In [10]:
order_items.head()


,order_item_id,order_id,product_id,quantity,unit_price
0,1,1,100,3,102000
1,2,1,87,5,25000
2,3,1,7,3,142000
3,4,1,9,3,193000
4,5,2,72,4,189000


앞부분만 보고 전체 데이터가 정상이라고 판단하기는 어렵습니다. `tail()`로 마지막 부분도 확인해 봅니다.


In [11]:
customers.tail()


,customer_id,name,gender,age,city,signup_date
145,146,고준영,M,61,성남,2023-12-19
146,147,김예은,M,19,부산,2025-07-15
147,148,김준혁,M,29,고양,2024-04-02
148,149,안성현,M,20,부산,2026-05-07
149,150,이경숙,M,40,대전,2026-07-21


### 생각해 보기

`head()`와 `tail()`을 보면서 아래를 확인해 보세요.

- 날짜처럼 보이는 컬럼이 있나요?
- 숫자처럼 보이는 컬럼이 있나요?
- ID처럼 보이는 컬럼이 있나요?
- 사람이 직접 읽을 수 있는 이름이나 범주 값이 있나요?


## 7. 컬럼명 확인하기

컬럼명은 코드 작성에서 매우 중요합니다. 실제 컬럼명이 `customer_id`인데 LLM이나 사람이 `cust_id`라고 쓰면 코드는 실행되지 않습니다.


In [12]:
for name, df in datasets.items():
    print(f"[{name}]")
    print(list(df.columns))
    print()


[customers]
['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

[products]
['product_id', 'product_name', 'category', 'price']

[orders]
['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']

[order_items]
['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']



In [13]:
column_summary = pd.DataFrame([
    {
        "dataset": name,
        "column_count": len(df.columns),
        "column_names": ", ".join(df.columns),
    }
    for name, df in datasets.items()
])

column_summary


,dataset,column_count,column_names
0,customers,6,"customer_id, name, gender, age, city, signup_date"
1,products,4,"product_id, product_name, category, price"
2,orders,5,"order_id, customer_id, order_date, payment_met..."
3,order_items,5,"order_item_id, order_id, product_id, quantity,..."


### 생각해 보기

- 고객을 구분하는 컬럼은 무엇인가요?
- 주문을 구분하는 컬럼은 무엇인가요?
- 상품을 구분하는 컬럼은 무엇인가요?
- 여러 파일을 연결할 때 사용할 수 있을 것 같은 컬럼은 무엇인가요?


## 8. 데이터 타입 확인하기

`info()`는 컬럼별 데이터 타입과 비어 있지 않은 값의 개수를 보여 줍니다.

특히 날짜처럼 보이지만 `object`로 저장된 컬럼을 주의해서 봅니다. pandas에서 `object`는 보통 문자열 또는 여러 타입이 섞인 컬럼일 때 나타납니다.


In [14]:
customers.info()


<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  150 non-null    int64
 1   name         150 non-null    str  
 2   gender       150 non-null    str  
 3   age          150 non-null    int64
 4   city         150 non-null    str  
 5   signup_date  150 non-null    str  
dtypes: int64(2), str(4)
memory usage: 11.0 KB


In [15]:
for name, df in datasets.items():
    print(f"\n===== {name} =====")
    df.info()



===== customers =====
<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  150 non-null    int64
 1   name         150 non-null    str  
 2   gender       150 non-null    str  
 3   age          150 non-null    int64
 4   city         150 non-null    str  
 5   signup_date  150 non-null    str  
dtypes: int64(2), str(4)
memory usage: 11.0 KB

===== products =====
<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   product_id    100 non-null    int64
 1   product_name  100 non-null    str  
 2   category      100 non-null    str  
 3   price         100 non-null    int64
dtypes: int64(2), str(2)
memory usage: 6.0 KB

===== orders =====
<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 5 columns)

In [16]:
dtype_summary = pd.concat(
    [df.dtypes.rename(name) for name, df in datasets.items()],
    axis=1,
).fillna("")

dtype_summary


,customers,products,orders,order_items
customer_id,int64,,int64,
name,str,,,
gender,str,,,
age,int64,,,
city,str,,,
signup_date,str,,,
product_id,,int64,,int64
product_name,,str,,
category,,str,,
price,,int64,,


### 생각해 보기

- 숫자형 컬럼은 어떤 것들이 있나요?
- 문자형 컬럼은 어떤 것들이 있나요?
- 날짜처럼 보이지만 아직 문자열일 가능성이 있는 컬럼은 무엇인가요?


![데이터 구조 점검 흐름도](../book/assets/images/ch03/ch03_data_check_flow.svg)

데이터 구조 점검은 파일 확인, 로드, 크기 확인, 컬럼 확인, 타입 확인, 결측치 확인, 중복 확인, 키 관계 확인 순서로 진행하면 좋습니다.


### 실행/결과
4개 CSV 로딩 여부: O
각 데이터 shape: customers: (150, 6), products: (100, 4), orders: (300, 5), order_items: (764, 5)
주요 컬럼: 
  - customers: 'customer_id', 'name', 'gender', 'age', 'city', 'signup_date'
  - products: 'product_id', 'product_name', 'category', 'price'
  - orders: 'order_id', 'customer_id', 'order_date', 'payment_method', 'order_status'
  - order_items: 'order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price'
dtypes에서 주목한 컬럼: 'customers.signup_date'와 'orders.order_date'는 날짜처럼 보이지만 처음 불러왔을 때는 문자열(str)이었다. 'age', 'price', 'quantity', 'unit_price'와 각 ID는 int64였다.

### Evidence
![데이터 구조 확인](images/step01_structure.png)

### 결과 관찰
네 파일은 모두 정상적으로 로딩되었다. 가장 행이 많은 데이터는 order_items로 764행이고, orders는 300행이었다. head()와 tail()에서 컬럼과 값이 정상적으로 표시되었으며, customers에는 고객 이름, 성별, 나이, 지역, 가입일이 포함되어 있었다. order_items는 한 주문 안의 개별 상품 항목을 나타내므로 주문 한 건에 여러 상품이 포함되면 orders보다 행 수가 많아질 수 있다.

### 나의 해석과 판단
가장 먼저 주의할 데이터는 customers이다. name은 실제 데이터라면 개인을 직접 식별할 수 있는 정보이고, customer_id는 다른 테이블의 주문 기록과 고객 정보를 연결하는 식별자이다. 또한 gender, age, city, signup_date도 단독으로는 직접 식별정보가 아니더라도 여러 속성이 결합되면 개인을 구분하는 데 사용될 수 있다. 이번 데이터는 노트북에서 가상의 온라인 쇼핑몰 샘플 데이터로 설명되어 있지만, 실제 업무 데이터라면 LLM이나 외부 분석 도구에 원본 고객명이나 개별 고객 행을 그대로 전달하지 않고 구조·집계 정보만 제공해야 한다.

데이터 구조 측면에서는 파일마다 한 행의 의미가 다르다는 점도 중요하다. order_items의 764행을 주문 764건으로 해석하면 주문 수를 과대 계산하게 된다. 또한 날짜 의미를 가진 signup_date와 order_date가 처음에는 문자열이므로 기간 분석 전에 날짜 타입으로 변환해야 하며, 금액을 집계할 때는 order_status에 따라 어떤 주문을 포함할지도 정해야 한다.

### 업무·분석적 의미
데이터 구조와 식별자 역할을 확인하지 않고 분석하면 주문 수와 주문상품 수를 혼동하거나, 조인 과정에서 행과 금액이 의도치 않게 증가할 수 있다. 문자열 날짜를 그대로 사용하면 기간 계산이 잘못되거나 실패할 수 있고, ID를 일반 숫자처럼 분석하면 의미 없는 평균이나 분포를 만들 수 있다. 개인정보가 포함된 실제 데이터라면 원본 값을 그대로 LLM에 입력하는 과정 자체가 정보 노출 위험이 될 수 있으므로 분석 단계부터 최소한의 정보만 사용하는 원칙이 필요하다.

### 한계와 추가 확인 사항
이 단계에서는 데이터 크기, 컬럼명, 데이터 타입과 일부 행만 확인했기 때문에 값의 품질까지 판단할 수 없다. 결측치, 중복, 숫자 범위, 범주 값의 표기, 날짜 변환 실패 여부와 파일 간 키 관계는 별도 점검이 필요하다. 또한 현재 데이터가 가상 샘플이므로 실제 개인정보가 포함되었다고 볼 수는 없지만, 동일한 스키마를 실제 고객 데이터에 적용할 경우 어떤 컬럼을 마스킹·제외할지 별도 기준이 필요하다.


## 9. 결측치 확인하기

결측치는 값이 비어 있는 상태입니다. 결측치가 있으면 평균, 비율, 그룹별 집계 결과가 달라질 수 있습니다.

`isna().sum()`은 컬럼별 결측치 개수를 계산합니다.


In [17]:
customers.isna().sum()


customer_id    0
name           0
gender         0
age            0
city           0
signup_date    0
dtype: int64

In [18]:
missing_summary = pd.concat(
    [df.isna().sum().rename(name) for name, df in datasets.items()],
    axis=1,
).fillna("").astype(str)

missing_summary


,customers,products,orders,order_items
customer_id,0.0,,0.0,
name,0.0,,,
gender,0.0,,,
age,0.0,,,
city,0.0,,,
signup_date,0.0,,,
product_id,,0.0,,0.0
product_name,,0.0,,
category,,0.0,,
price,,0.0,,


In [19]:
missing_rate_summary = pd.concat(
    [(df.isna().mean() * 100).round(2).rename(name) for name, df in datasets.items()],
    axis=1,
).fillna("")

missing_rate_summary


,customers,products,orders,order_items
customer_id,0.0,,0.0,
name,0.0,,,
gender,0.0,,,
age,0.0,,,
city,0.0,,,
signup_date,0.0,,,
product_id,,0.0,,0.0
product_name,,0.0,,
category,,0.0,,
price,,0.0,,


### 결측치 해석 팁

결측치가 있다고 해서 무조건 삭제하는 것은 아닙니다.

| 처리 방법 | 설명 |
| --- | --- |
| 행 제외 | 결측치가 있는 행을 분석에서 제외합니다. |
| 대표값 대체 | 평균, 중앙값, 최빈값 등으로 채웁니다. |
| 별도 범주 처리 | `Unknown` 같은 범주로 표시합니다. |
| 컬럼 제외 | 분석 목적에 맞지 않는 컬럼은 사용하지 않습니다. |
| 원인 확인 | 수집 과정에서 문제가 있었는지 확인합니다. |


## 10. 중복 데이터 확인하기

중복은 같은 행이나 같은 ID가 반복되는 상태입니다.

단, 모든 중복이 오류는 아닙니다. 예를 들어 `order_items`에서는 한 주문에 여러 상품이 들어갈 수 있으므로 같은 `order_id`가 여러 번 나올 수 있습니다.


In [10]:
duplicate_rows = pd.DataFrame([
    {
        "dataset": name,
        "duplicated_rows": df.duplicated().sum(),
    }
    for name, df in datasets.items()
])

duplicate_rows


,dataset,duplicated_rows
0,customers,0
1,products,0
2,orders,0
3,order_items,0


In [11]:
id_duplicate_checks = pd.DataFrame([
    {
        "check": "customers.customer_id",
        "duplicated_count": customers["customer_id"].duplicated().sum(),
        "interpretation": "0이어야 고객 ID가 유일합니다.",
    },
    {
        "check": "products.product_id",
        "duplicated_count": products["product_id"].duplicated().sum(),
        "interpretation": "0이어야 상품 ID가 유일합니다.",
    },
    {
        "check": "orders.order_id",
        "duplicated_count": orders["order_id"].duplicated().sum(),
        "interpretation": "0이어야 주문 ID가 유일합니다.",
    },
    {
        "check": "order_items.order_id",
        "duplicated_count": order_items["order_id"].duplicated().sum(),
        "interpretation": "한 주문에 여러 상품이 있으면 0보다 클 수 있습니다.",
    },
])

id_duplicate_checks


,check,duplicated_count,interpretation
0,customers.customer_id,0,0이어야 고객 ID가 유일합니다.
1,products.product_id,0,0이어야 상품 ID가 유일합니다.
2,orders.order_id,0,0이어야 주문 ID가 유일합니다.
3,order_items.order_id,464,한 주문에 여러 상품이 있으면 0보다 클 수 있습니다.


### 생각해 보기

- `customers.customer_id` 중복과 `order_items.order_id` 중복은 왜 의미가 다를까요?
- 중복 개수만 보고 삭제하면 위험한 이유는 무엇일까요?


주요 ID 결측: 네 데이터셋의 전체 컬럼에서 결측치가 0건으로 확인되어, Notebook에서 확인한 주요 ID 컬럼의 결측도 0건이었다.
주요 ID 중복: customers.customer_id 0건, products.product_id 0건, orders.order_id 0건이었다. order_items.order_id는 464행에서 반복되었지만 이는 주문상세의 외래 키이므로 한 주문에 여러 상품이 있을 때 정상적으로 반복될 수 있다.
전체 행 중복: customers, products, orders, order_items 모두 0건이었다.

### Evidence2
![Evidence2](images/step02_quality.png)

### 결과 관찰

isna().sum()과 결측 비율 확인 결과 네 데이터셋의 모든 컬럼에서 결측치는 0건이었다. 전체 행이 완전히 동일한 중복도 네 데이터셋 모두 0건이었다. 고객·상품·주문을 구분하는 customer_id, product_id, order_id에서는 중복이 발견되지 않았지만, order_items.order_id에서는 464개의 중복 행이 확인되었다.

### 나의 해석과 판단

먼저 확인해야 할 것은 기본키 역할을 하는 ID의 결측과 중복이다. 기본키가 비어 있거나 중복되면 다른 데이터와 연결할 때 행이 잘못 합쳐지거나 누락될 수 있다. 반면 order_items.order_id처럼 부모 주문을 참조하는 외래 키의 반복은 한 주문에 여러 상품이 들어가는 구조상 정상일 수 있으므로 단순히 duplicated() 결과가 0보다 크다는 이유만으로 삭제하면 안 된다.

### 업무·분석적 의미

order_items.order_id의 반복을 데이터 오류로 판단해 삭제하면 한 주문에 포함된 정상적인 상품 항목이 사라지고 주문 금액도 실제보다 작아질 수 있다. 따라서 중복 여부는 숫자만 보는 것이 아니라 해당 컬럼이 기본키인지 외래 키인지, 그리고 한 행이 무엇을 의미하는지를 함께 보고 판단해야 한다.

### 한계와 추가 확인 사항

현재 Notebook에서는 customers.customer_id, products.product_id, orders.order_id, order_items.order_id에 대한 중복을 직접 확인했다. 하지만 주문상세 자체의 기본키로 보이는 order_items.order_item_id의 고유성은 별도의 중복 체크 코드로 확인하지 않았다. 또한 isna()로 잡히지 않는 unknown, -, 빈 문자열 같은 대체 결측 표기가 있는지는 현재 결과만으로 단정할 수 없다.

## 11. 숫자형 컬럼 기본 통계 확인하기

`describe()`는 숫자형 컬럼의 개수, 평균, 표준편차, 최솟값, 사분위수, 최댓값을 보여 줍니다.

최솟값이나 최댓값이 지나치게 이상하면 데이터 오류나 이상치 가능성을 의심할 수 있습니다.


In [22]:
customers.describe()


,customer_id,age
count,150.000000,150.000000
mean,75.500000,42.086667
std,43.445368,15.613166
min,1.000000,19.000000
25%,38.250000,29.000000
50%,75.500000,40.000000
75%,112.750000,57.000000
max,150.000000,69.000000


In [23]:
products[["price"]].describe()


,price
count,100.000000
mean,110040.000000
std,56433.910574
min,5000.000000
25%,65750.000000
50%,112000.000000
75%,161000.000000
max,200000.000000


In [24]:
order_items[["quantity", "unit_price"]].describe()


,quantity,unit_price
count,764.000000,764.000000
mean,3.053665,108561.518325
std,1.410873,56996.770604
min,1.000000,5000.000000
25%,2.000000,62000.000000
50%,3.000000,111000.000000
75%,4.000000,161250.000000
max,5.000000,200000.000000


숫자가 문자열로 저장된 경우도 있습니다. 예를 들어 `"10,000"`처럼 쉼표가 포함된 문자열은 바로 계산하기 어렵습니다.


In [25]:
price_text = pd.Series(["10,000", "25,500", "3000", "확인필요"])
price_number = pd.to_numeric(
    price_text.str.replace(",", "", regex=False),
    errors="coerce",
)

pd.DataFrame({
    "original": price_text,
    "converted": price_number,
})


,original,converted
0,"10,000",10000.0
1,"25,500",25500.0
2,3000,3000.0
3,확인필요,NaN


`errors="coerce"`는 숫자로 바꿀 수 없는 값을 `NaN`으로 처리합니다. 변환 후에는 새로 생긴 결측치가 있는지도 확인해야 합니다.


## 12. 범주형 컬럼 고유값 확인하기

문자형 또는 범주형 컬럼은 고유값 개수와 빈도를 확인합니다. 예를 들어 지역, 성별, 카테고리, 주문 상태 같은 컬럼은 `value_counts()`로 분포를 볼 수 있습니다.


In [26]:
customers["city"].value_counts().head(10)


city
성남    21
광주    17
부산    16
대구    15
서울    15
울산    14
인천    14
대전    14
수원    13
고양    11
Name: count, dtype: int64

In [27]:
products["category"].value_counts()


category
스포츠     19
전자기기    17
생활용품    16
뷰티      16
도서      14
패션      11
식품       7
Name: count, dtype: int64

In [28]:
orders["order_status"].value_counts()


order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

In [29]:
categorical_summary = pd.DataFrame([
    {"dataset": "customers", "column": "city", "unique_count": customers["city"].nunique()},
    {"dataset": "customers", "column": "gender", "unique_count": customers["gender"].nunique()},
    {"dataset": "products", "column": "category", "unique_count": products["category"].nunique()},
    {"dataset": "orders", "column": "payment_method", "unique_count": orders["payment_method"].nunique()},
    {"dataset": "orders", "column": "order_status", "unique_count": orders["order_status"].nunique()},
])

categorical_summary


,dataset,column,unique_count
0,customers,city,10
1,customers,gender,2
2,products,category,7
3,orders,payment_method,4
4,orders,order_status,3


### 생각해 보기

- 특정 값에 데이터가 지나치게 몰려 있나요?
- 오타나 표기 차이처럼 보이는 값이 있나요?
- 나중에 그룹별 분석 기준으로 쓰기 좋은 컬럼은 무엇인가요?


## 13. 날짜 컬럼 확인하기

날짜 컬럼은 월별, 요일별, 기간별 분석에 자주 사용됩니다.

하지만 CSV에서 읽어온 날짜는 처음에는 문자열(`object`)일 수 있습니다. `pd.to_datetime()`으로 날짜 타입으로 바꿔야 날짜 계산을 안전하게 할 수 있습니다.


In [30]:
orders["order_date"].head()


0    2025-10-16
1    2026-06-05
2    2026-03-12
3    2026-06-19
4    2026-05-25
Name: order_date, dtype: str

In [31]:
print("변환 전 타입:", orders["order_date"].dtype)

orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")

print("변환 후 타입:", orders["order_date"].dtype)
print("날짜 변환 실패 건수:", orders["order_date"].isna().sum())
print("가장 빠른 주문일:", orders["order_date"].min())
print("가장 최근 주문일:", orders["order_date"].max())


변환 전 타입: str
변환 후 타입: datetime64[us]
날짜 변환 실패 건수: 0
가장 빠른 주문일: 2025-09-15 00:00:00
가장 최근 주문일: 2026-09-15 00:00:00


In [32]:
orders.assign(
    order_year=orders["order_date"].dt.year,
    order_month=orders["order_date"].dt.month,
    order_day_name=orders["order_date"].dt.day_name(),
).head()


,order_id,customer_id,order_date,payment_method,order_status,order_year,order_month,order_day_name
0,1,123,2025-10-16,card,completed,2025,10,Thursday
1,2,77,2026-06-05,naver_pay,cancelled,2026,6,Friday
2,3,138,2026-03-12,bank_transfer,cancelled,2026,3,Thursday
3,4,57,2026-06-19,kakao_pay,cancelled,2026,6,Friday
4,5,125,2026-05-25,card,cancelled,2026,5,Monday


### 생각해 보기

- 데이터는 어느 기간을 포함하고 있나요?
- 월별 매출 분석을 하기에 충분한 기간인가요?
- 날짜 변환 실패 건수가 0보다 크다면 무엇을 확인해야 할까요?


### Evidence3

숫자형 범위에서 주목한 값: customers.age 19~69세, products.price 5,000~200,000, order_items.quantity 1~5, order_items.unit_price 5,000~200,000 범위였다.
범주형 빈도에서 주목한 값: 상품 카테고리는 스포츠 19개가 가장 많았다. 주문상태는 completed 184건, cancelled 64건, refunded 52건이었다. 고객 도시는 성남 21명으로 가장 많았다.
날짜 변환 실패 건수: orders.order_date 0건.
날짜 범위: orders.order_date는 2025-09-15부터 2026-09-15까지였다.

![Evidence3](images/step03_distribution.png)

### 결과 관찰

숫자형 컬럼의 기초 통계를 확인한 결과 나이는 19~69세, 상품 가격과 주문 단가는 5,000~200,000, 주문 수량은 1~5 범위였다. orders.order_date는 문자열에서 datetime64 타입으로 변환되었고 변환 실패는 0건이었다. 주문상태에는 완료 주문 184건뿐 아니라 취소 64건과 환불 52건도 함께 포함되어 있었다.

### 나의 해석과 판단

현재 관찰된 숫자 범위만으로는 값이 잘못되었다고 판단하기 어렵다. 예를 들어 19세와 69세가 모두 서비스 대상이 될 수 있는지, 200,000이라는 가격이 정상 범위인지, 한 상품을 최대 5개까지 주문할 수 있는지는 실제 업무 규칙과 비교해야 판단할 수 있다. cancelled와 refunded 역시 데이터 오류가 아니라 정상적인 주문 상태일 수 있으므로 삭제 대상이 아니라 분석 목적에 따라 포함 여부를 정해야 한다.

### 업무·분석적 의미

숫자 범위와 범주 분포를 먼저 확인하면 잘못 입력된 값이나 특정 값의 과도한 편중이 평균·합계·그룹별 분석을 왜곡하는 것을 줄일 수 있다. 특히 현재 주문 데이터에는 취소와 환불이 총 116건 포함되어 있으므로 모든 주문의 quantity × unit_price를 더한 값은 확정 매출과 다를 수 있다. 날짜를 정상적으로 변환해 두면 이후 월별·기간별 분석도 안전하게 수행할 수 있다.

### 한계와 추가 확인 사항

현재 Notebook에서 실제 날짜 변환과 범위 확인을 수행한 컬럼은 orders.order_date이다. customers.signup_date도 날짜 후보이지만 아직 동일한 방식으로 변환 실패와 범위를 검증하지 않았다. 또한 가격의 통화 단위, 할인·세금·부분 환불 여부, 주문상태별 매출 인정 기준은 현재 데이터만으로 확인할 수 없다.



## 14. 여러 파일의 관계 확인하기

온라인 쇼핑몰 데이터는 고객, 상품, 주문, 주문 상세 데이터가 서로 연결되어야 분석할 수 있습니다.

| 연결 관계 | 의미 |
| --- | --- |
| `customers.customer_id` ↔ `orders.customer_id` | 어떤 고객이 주문했는지 연결합니다. |
| `orders.order_id` ↔ `order_items.order_id` | 주문과 주문 상세를 연결합니다. |
| `products.product_id` ↔ `order_items.product_id` | 주문 상세와 상품 정보를 연결합니다. |


![4개 CSV 파일 간 키 관계도](../book/assets/images/ch03/ch03_csv_key_relationships.svg)


In [33]:
invalid_customers = orders[~orders["customer_id"].isin(customers["customer_id"])]
invalid_orders = order_items[~order_items["order_id"].isin(orders["order_id"])]
invalid_products = order_items[~order_items["product_id"].isin(products["product_id"])]

relationship_check = pd.DataFrame([
    {
        "relationship": "orders.customer_id -> customers.customer_id",
        "invalid_rows": len(invalid_customers),
    },
    {
        "relationship": "order_items.order_id -> orders.order_id",
        "invalid_rows": len(invalid_orders),
    },
    {
        "relationship": "order_items.product_id -> products.product_id",
        "invalid_rows": len(invalid_products),
    },
])

relationship_check


,relationship,invalid_rows
0,orders.customer_id -> customers.customer_id,0
1,order_items.order_id -> orders.order_id,0
2,order_items.product_id -> products.product_id,0


`invalid_rows`가 모두 0이면 샘플 데이터에서는 기본적인 연결 관계가 유지되고 있다고 볼 수 있습니다. 0보다 큰 값이 있다면 어느 파일에서 기준 ID가 빠져 있는지 먼저 확인해야 합니다.


## 15. 간단한 병합으로 관계 확인하기

키 관계가 맞는지 확인한 뒤에는 데이터를 병합해 볼 수 있습니다. 아래 코드는 주문 상세(`order_items`)에 상품 정보(`products`)를 붙이고, 각 행의 금액을 계산합니다.


In [34]:
order_items_with_products = order_items.merge(
    products,
    on="product_id",
    how="left",
)

order_items_with_products["line_amount"] = (
    order_items_with_products["quantity"] * order_items_with_products["unit_price"]
)

order_items_with_products.head()


,order_item_id,order_id,product_id,quantity,unit_price,product_name,category,price,line_amount
0,1,1,100,3,102000,도서 상품 100,도서,102000,306000
1,2,1,87,5,25000,도서 상품 087,도서,25000,125000
2,3,1,7,3,142000,도서 상품 007,도서,142000,426000
3,4,1,9,3,193000,스포츠 상품 009,스포츠,193000,579000
4,5,2,72,4,189000,뷰티 상품 072,뷰티,189000,756000


In [35]:
category_sales = (
    order_items_with_products
    .groupby("category", as_index=False)["line_amount"]
    .sum()
    .sort_values("line_amount", ascending=False)
)

category_sales


,category,line_amount
3,스포츠,50174000
1,뷰티,47551000
5,전자기기,41003000
2,생활용품,34839000
4,식품,33597000
0,도서,24645000
6,패션,23801000


이 결과는 본격적인 EDA가 아니라, 데이터 관계가 실제로 연결되는지 확인하는 작은 점검입니다. 분석 결론을 내리기 전에 결측치, 주문 상태, 취소 주문 처리 기준 등을 더 확인해야 합니다.


## 16. 반복 점검을 함수로 정리하기

여러 데이터셋에 같은 점검을 반복할 때는 함수로 정리하면 편합니다.


In [36]:
def check_data_overview(name: str, df: pd.DataFrame) -> None:
    print(f"===== {name} =====")
    print("shape:", df.shape)
    print("\ncolumns:")
    print(list(df.columns))
    print("\ndtypes:")
    print(df.dtypes)
    print("\nmissing values:")
    print(df.isna().sum())
    print("\nduplicated rows:", df.duplicated().sum())


check_data_overview("customers", customers)


===== customers =====
shape: (150, 6)

columns:
['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

dtypes:
customer_id    int64
name             str
gender           str
age            int64
city             str
signup_date      str
dtype: object

missing values:
customer_id    0
name           0
gender         0
age            0
city           0
signup_date    0
dtype: int64

duplicated rows: 0


In [37]:
for name, df in datasets.items():
    check_data_overview(name, df)
    print()


===== customers =====
shape: (150, 6)

columns:
['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

dtypes:
customer_id    int64
name             str
gender           str
age            int64
city             str
signup_date      str
dtype: object

missing values:
customer_id    0
name           0
gender         0
age            0
city           0
signup_date    0
dtype: int64

duplicated rows: 0

===== products =====
shape: (100, 4)

columns:
['product_id', 'product_name', 'category', 'price']

dtypes:
product_id      int64
product_name      str
category          str
price           int64
dtype: object

missing values:
product_id      0
product_name    0
category        0
price           0
dtype: int64

duplicated rows: 0

===== orders =====
shape: (300, 5)

columns:
['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']

dtypes:
order_id                   int64
customer_id                int64
order_date        datetime64[us]
payment_method          

![Jupyter Notebook 데이터 구조 점검 결과 화면 예시](../book/assets/images/ch03/ch03_jupyter_data_overview_result.svg)


### Evidence4
없는 customer_id: 0건
없는 order_id: 0건
없는 product_id: 0건
![Evidence4](images/step04_relationship.png)

### 결과 관찰

orders.customer_id 중 customers.customer_id에 없는 값은 0건이었다. order_items.order_id 중 orders.order_id에 없는 값도 0건이었고, order_items.product_id 중 products.product_id에 없는 값 역시 0건이었다. 따라서 Notebook에서 확인한 세 참조 관계는 모두 연결 가능한 상태였다.

### 나의 해석과 판단

현재 데이터에서는 기본적인 파일 간 참조 관계가 유지되고 있다고 판단할 수 있다. 하지만 만약 부모 데이터에 존재하지 않는 키가 발견되더라도 해당 행을 바로 삭제하면 안 된다. 데이터 추출 시점 차이, 부모 파일 누락, 기간 필터 차이, 데이터 이관 문제 등 원인이 있을 수 있으므로 먼저 발생 원인과 영향을 받는 행의 범위를 확인해야 한다.

### 업무·분석적 의미

키 관계를 사전에 검증하면 여러 CSV를 병합할 때 어떤 행이 누락되거나 의도치 않게 늘어나는 문제를 예방할 수 있다. 실제 Notebook에서는 order_items와 products를 product_id로 병합한 뒤 quantity × unit_price로 line_amount를 계산하고 카테고리별 금액을 집계할 수 있음을 확인했다. 다만 이 금액은 주문상태를 반영하기 전의 주문상세 금액이다.

### 한계와 추가 확인 사항

참조 키가 모두 존재한다는 것은 테이블 간 연결이 가능하다는 의미일 뿐, 각 거래의 내용까지 업무적으로 올바르다는 의미는 아니다. 또한 Notebook의 병합 코드에서는 병합 전후 행 수가 정확히 같은지 별도 숫자로 검증하지 않았기 때문에, 실제 분석 단계에서는 병합 전후 행 수와 조인 방식까지 함께 확인하는 것이 안전하다.

## 17. LLM에게 데이터 구조를 설명시키는 법

LLM에게 원본 데이터를 그대로 붙여 넣는 것은 피하는 것이 좋습니다. 대신 아래처럼 구조 요약만 전달합니다.

- 파일명
- 컬럼명
- 행과 열 개수
- 데이터 타입
- 결측치 개수
- 중복 여부
- 파일 간 키 관계


In [38]:
llm_dataset_summary = shape_summary.merge(column_summary, on="dataset")
llm_dataset_summary


,dataset,rows,columns,column_count,column_names
0,customers,150,6,6,"customer_id, name, gender, age, city, signup_date"
1,products,100,4,4,"product_id, product_name, category, price"
2,orders,300,5,5,"order_id, customer_id, order_date, payment_met..."
3,order_items,764,5,5,"order_item_id, order_id, product_id, quantity,..."


In [39]:
for _, row in llm_dataset_summary.iterrows():
    print(f"- {row['dataset']}: {row['rows']}행 {row['columns']}열")
    print(f"  컬럼: {row['column_names']}")


- customers: 150행 6열
  컬럼: customer_id, name, gender, age, city, signup_date
- products: 100행 4열
  컬럼: product_id, product_name, category, price
- orders: 300행 5열
  컬럼: order_id, customer_id, order_date, payment_method, order_status
- order_items: 764행 5열
  컬럼: order_item_id, order_id, product_id, quantity, unit_price


### 데이터 구조 설명 요청 예시

아래 프롬프트는 LLM에게 붙여 넣을 수 있는 예시입니다. 실제 데이터 전체가 아니라 구조 정보만 포함합니다.


In [40]:
prompt = f"""
온라인 쇼핑몰 데이터 분석을 시작하기 전에 다음 CSV 파일들의 구조를 이해하려고 합니다.

데이터셋 요약:
{llm_dataset_summary[['dataset', 'rows', 'columns', 'column_names']].to_string(index=False)}

파일 간 관계:
- customers.customer_id -> orders.customer_id
- orders.order_id -> order_items.order_id
- products.product_id -> order_items.product_id

요청:
1. 각 파일이 어떤 역할을 하는지 설명해 주세요.
2. 분석 전에 확인해야 할 항목을 체크리스트로 정리해 주세요.
3. 실제 데이터 확인 없이 단정한 내용과 추가 확인이 필요한 내용을 구분해 주세요.
"""

print(prompt)



온라인 쇼핑몰 데이터 분석을 시작하기 전에 다음 CSV 파일들의 구조를 이해하려고 합니다.

데이터셋 요약:
    dataset  rows  columns                                                    column_names
  customers   150        6               customer_id, name, gender, age, city, signup_date
   products   100        4                       product_id, product_name, category, price
     orders   300        5 order_id, customer_id, order_date, payment_method, order_status
order_items   764        5       order_item_id, order_id, product_id, quantity, unit_price

파일 간 관계:
- customers.customer_id -> orders.customer_id
- orders.order_id -> order_items.order_id
- products.product_id -> order_items.product_id

요청:
1. 각 파일이 어떤 역할을 하는지 설명해 주세요.
2. 분석 전에 확인해야 할 항목을 체크리스트로 정리해 주세요.
3. 실제 데이터 확인 없이 단정한 내용과 추가 확인이 필요한 내용을 구분해 주세요.



## 18. LLM 답변 검증 연습

LLM이 다음과 같이 답했다고 가정해 봅니다.

> 고객 데이터에 age 컬럼이 있으므로 연령대별 매출 분석을 바로 수행하면 됩니다.

이 답변은 그럴듯하지만 충분히 안전하지 않습니다. 아래 내용을 직접 확인해야 합니다.

- `age` 컬럼이 실제로 존재하는가?
- `age` 컬럼에 결측치나 이상치가 있는가?
- 고객 데이터와 주문 데이터가 `customer_id`로 연결되는가?
- 매출을 계산하려면 주문 상세와 상품 또는 단가 정보가 필요한가?
- 취소 주문을 포함할지 제외할지 기준이 있는가?


In [41]:
validation_check = pd.DataFrame([
    {
        "question": "customers에 age 컬럼이 있는가?",
        "result": "age" in customers.columns,
    },
    {
        "question": "age 결측치 개수는?",
        "result": customers["age"].isna().sum() if "age" in customers.columns else "컬럼 없음",
    },
    {
        "question": "orders.customer_id가 customers.customer_id와 연결되는가?",
        "result": len(invalid_customers) == 0,
    },
    {
        "question": "매출 계산에 필요한 quantity와 unit_price가 있는가?",
        "result": {"quantity", "unit_price"}.issubset(order_items.columns),
    },
])

validation_check


,question,result
0,customers에 age 컬럼이 있는가?,True
1,age 결측치 개수는?,0
2,orders.customer_id가 customers.customer_id와 연결되는가?,True
3,매출 계산에 필요한 quantity와 unit_price가 있는가?,True


LLM에 제공한 Safe Context: 데이터셋명, 행·열 개수, 컬럼명과 파일 간 키 관계만 구조 요약으로 제공했다. 실제 고객 이름이나 개별 고객 행의 원본 값은 프롬프트에 포함하지 않았다.

LLM이 제안한 추가 점검: Notebook에서는 “customers에 age 컬럼이 있으므로 연령대별 매출 분석을 바로 수행하면 된다”는 LLM의 가상 답변을 검증 대상으로 두었다. 이에 따라 age 컬럼 존재 여부, age 결측치, 고객-주문 키 연결, 금액 계산에 필요한 quantity와 unit_price 존재 여부를 확인하도록 했다. 또한 취소 주문을 매출에 포함할지 여부도 추가 확인 사항으로 제시했다.

실제 데이터에서 확인한 항목: age 컬럼 존재 True, age 결측치 0건, orders.customer_id와 customers.customer_id 연결 가능 True, order_items에 quantity와 unit_price 존재 True로 확인되었다.

채택/수정/보류한 내용: 실제 컬럼과 키 관계를 코드로 다시 확인하는 방법은 채택했다. 반면 age 컬럼이 있다는 이유만으로 연령대별 매출 분석을 바로 수행할 수 있다는 결론은 수정했다. 취소·환불 주문을 매출에 포함할지 여부는 업무 기준이 필요하므로 보류했다.

### Evidence5
![Evidence5](images/step05_llm.png)

### 나의 해석과 판단

LLM 제안 중 가장 유용한 부분은 분석 아이디어를 실제 컬럼·결측·키 관계와 대조해 다시 확인하도록 한 점이다. 반대로 가장 조심해야 할 부분은 컬럼 하나가 존재한다는 사실만으로 필요한 데이터 조건이 모두 충족됐다고 단정하는 것이다. 연령대별 매출을 계산하려면 고객과 주문이 정상적으로 연결되어야 하고, 주문상세의 수량·단가가 필요하며, 완료·취소·환불 주문을 어떻게 처리할지도 정해야 한다.

Safe Context 측면에서도 구조 정보와 집계 결과만 제공한 방식이 적절하다고 판단했다. 실제 고객 데이터였다면 name과 같은 직접 식별정보나 개별 customer_id 값이 포함된 원본 행을 LLM에 그대로 전달하기보다 분석에 필요한 최소한의 스키마·집계 정보만 제공하는 것이 안전하다.

### 한계와 추가 확인 사항

현재 Notebook은 실제 LLM API를 호출한 결과를 저장한 것이 아니라, LLM이 할 수 있는 가상의 답변을 실제 데이터와 비교해 검증하는 연습 형태이다. 따라서 실제 LLM을 사용한 과정을 Evidence로 남기려면 사용한 프롬프트, 받은 제안, 실제 검증 결과, 채택·수정·보류 판단을 구분해 기록하는 것이 더 명확하다. 또한 이번 데이터는 가상 샘플이므로 식별정보 관련 판단은 실제 개인정보 노출 사례가 아니라 실제 업무 데이터에 적용할 안전 원칙으로 해석해야 한다.

## 19. 이번 장 점검 체크리스트

| 점검 항목 | 확인 |
| --- | --- |
| 필요한 CSV 파일이 모두 존재하는가? | ☑ |
| 각 데이터셋의 행과 열 개수를 확인했는가? | ☑ |
| 컬럼명이 예상과 일치하는가? | ☑ |
| 날짜 컬럼의 데이터 타입을 확인했는가? | ☑ |
| 숫자 컬럼이 실제 숫자형으로 저장되어 있는가? | ☑ |
| 결측치가 있는 컬럼을 확인했는가? | ☑ |
| 중복 데이터가 있는지 확인했는가? | ☑ |
| 주요 ID 컬럼의 중복 여부를 확인했는가? | ☑ |
| 여러 파일을 연결할 키 컬럼을 확인했는가? | ☑ |
| 파일 간 키 관계가 실제로 연결 가능한지 확인했는가? | ☑ |
| LLM에 원본 데이터 대신 구조 요약만 입력했는가? | ☑ |
| LLM이 제안한 설명을 실제 데이터와 비교해 검증했는가? | ☑ |


## 20. 실습 과제

아래 과제를 직접 해결해 보세요.

1. 4개 CSV 파일의 행과 열 개수를 하나의 표로 정리하세요.
2. 각 파일의 결측치 개수와 결측치 비율을 확인하세요.
3. `orders.order_date`를 날짜 타입으로 변환하고 데이터 기간을 확인하세요.
4. `order_items`와 `products`를 병합해 카테고리별 주문 금액을 계산하세요.
5. LLM에게 데이터 구조 요약을 전달하는 프롬프트를 직접 작성하세요.
6. LLM이 만든 분석 아이디어가 실제 컬럼과 키 관계에 맞는지 검증하세요.


In [13]:
# 1. 4개 CSV 파일의 행과 열 개수
assignment_shape = pd.DataFrame([
    {"dataset": name, "rows": df.shape[0], "columns": df.shape[1]}
    for name, df in datasets.items()
])
print("[1] 데이터셋 크기")
print(assignment_shape.to_string(index=False))

# 2. 결측치 개수와 비율
missing_rows = []
for name, df in datasets.items():
    for column in df.columns:
        missing_rows.append({
            "dataset": name,
            "column": column,
            "missing_count": int(df[column].isna().sum()),
            "missing_rate_pct": round(float(df[column].isna().mean() * 100), 2),
        })
assignment_missing = pd.DataFrame(missing_rows)
print("\n[2] 결측치 개수와 비율")
print(assignment_missing.to_string(index=False))

# 3. 주문일을 날짜 타입으로 변환하고 기간 확인
orders["order_date"] = pd.to_datetime(orders["order_date"], errors="coerce")
print("\n[3] 주문일 점검")
print("dtype:", orders["order_date"].dtype)
print("변환 실패:", int(orders["order_date"].isna().sum()), "건")
print("기간:", orders["order_date"].min().date(), "~", orders["order_date"].max().date())

# 4. 주문상세와 상품을 병합해 카테고리별 주문 금액 계산
assignment_items = order_items.merge(
    products[["product_id", "category"]],
    on="product_id",
    how="left",
)
assignment_items["line_amount"] = assignment_items["quantity"] * assignment_items["unit_price"]
assignment_category_sales = (
    assignment_items.groupby("category", as_index=False)["line_amount"]
    .sum()
    .sort_values("line_amount", ascending=False)
)
print("\n[4] 카테고리별 주문 금액")
print(assignment_category_sales.to_string(index=False))

# 5. LLM에 전달할 Safe Context 기반 구조 요약 프롬프트
# 앞쪽 셀의 column_summary 변수에 의존하지 않도록 여기서 다시 만든다.
assignment_column_summary = pd.DataFrame([
    {
        "dataset": name,
        "column_names": ", ".join(df.columns.astype(str)),
    }
    for name, df in datasets.items()
])
assignment_llm_summary = assignment_shape.merge(assignment_column_summary, on="dataset")
assignment_prompt = f"""
온라인 쇼핑몰 데이터 분석을 시작하기 전에 다음 CSV 파일들의 구조를 이해하려고 합니다.

데이터셋 요약:
{assignment_llm_summary[['dataset', 'rows', 'columns', 'column_names']].to_string(index=False)}

파일 간 관계:
- customers.customer_id -> orders.customer_id
- orders.order_id -> order_items.order_id
- products.product_id -> order_items.product_id

요청:
1. 각 파일이 어떤 역할을 하는지 설명해 주세요.
2. 분석 전에 확인해야 할 항목을 체크리스트로 정리해 주세요.
3. 실제 데이터 확인 없이 단정한 내용과 추가 확인이 필요한 내용을 구분해 주세요.

주의: 실제 고객 이름이나 개별 고객 행 등 식별 가능한 원본 데이터는 사용하지 말고, 위 구조 요약만 사용해 주세요.
"""
print("\n[5] LLM 구조 설명 프롬프트")
print(assignment_prompt)

# 6. LLM 분석 아이디어 검증
# 가상 아이디어: "age 컬럼이 있으므로 연령대별 매출 분석을 바로 수행하면 된다."
# 앞쪽 셀의 invalid_customers 변수에도 의존하지 않도록 여기서 다시 계산한다.
assignment_invalid_customers = set(orders["customer_id"]) - set(customers["customer_id"])
assignment_validation = pd.DataFrame([
    {
        "check": "customers에 age 컬럼이 있는가?",
        "result": "age" in customers.columns,
    },
    {
        "check": "age 결측치가 0건인가?",
        "result": int(customers["age"].isna().sum()) == 0,
    },
    {
        "check": "orders.customer_id가 모두 customers에 존재하는가?",
        "result": len(assignment_invalid_customers) == 0,
    },
    {
        "check": "quantity와 unit_price가 모두 존재하는가?",
        "result": {"quantity", "unit_price"}.issubset(order_items.columns),
    },
    {
        "check": "주문상태가 하나뿐인가?",
        "result": orders["order_status"].nunique() == 1,
    },
])
print("\n[6] LLM 분석 아이디어 검증")
print(assignment_validation.to_string(index=False))
print("주문상태:", ", ".join(orders["order_status"].value_counts().index.astype(str)))
print("판단: 연령대별 금액 계산 자체는 가능하지만, completed/cancelled/refunded 중 어떤 상태를 매출에 포함할지 기준을 먼저 정해야 한다.")

# 19번 체크리스트 보완: 주문상세 기본키 고유성 추가 확인
order_item_id_duplicates = int(order_items["order_item_id"].duplicated().sum())
print("\n[추가 점검] order_items.order_item_id 중복:", order_item_id_duplicates, "건")

[1] 데이터셋 크기
    dataset  rows  columns
  customers   150        6
   products   100        4
     orders   300        5
order_items   764        5

[2] 결측치 개수와 비율
    dataset         column  missing_count  missing_rate_pct
  customers    customer_id              0               0.0
  customers           name              0               0.0
  customers         gender              0               0.0
  customers            age              0               0.0
  customers           city              0               0.0
  customers    signup_date              0               0.0
   products     product_id              0               0.0
   products   product_name              0               0.0
   products       category              0               0.0
   products          price              0               0.0
     orders       order_id              0               0.0
     orders    customer_id              0               0.0
     orders     order_date              0               0

## 마무리

이번 장의 핵심은 “데이터를 불러왔다”에서 끝내지 않는 것입니다.

데이터 분석을 시작하기 전에 파일, 행과 열, 컬럼명, 데이터 타입, 결측치, 중복, 키 관계를 확인해야 이후 분석이 흔들리지 않습니다. 다음 장에서는 이 구조를 바탕으로 pandas의 선택, 필터링, 정렬, 집계 기초를 다룹니다.
